# 🎮 Minecraft Mod 生成器 – 模型训练

本 Notebook 将在**免费 GPU** 上训练一个代码生成模型，用来根据中文描述生成 Minecraft Mod 代码。

## 使用步骤
1. 点击右上角 **“连接”**，选择 **T4 GPU** 运行时
2. 将 `mcmod_data.db` 上传到 Colab 环境（左侧文件图标 → 上传）
3. 依次执行下方代码单元格
4. 训练完成后，模型会自动保存到你的 **Google Drive**（或直接下载）

> 如果没有数据，请先在本地运行仓库中的 `collect_data.py` 生成数据库文件。

In [ ]:
# 挂载 Google Drive（用于保存模型，也可选择不挂载，最后直接下载）
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 安装依赖
!pip install -q torch transformers datasets peft pandas

In [ ]:
# 确认数据库文件已上传（如果已经在 Drive 里，可以改成对应路径）
import os
db_path = 'mcmod_data.db'
if not os.path.exists(db_path):
    # 尝试从 Google Drive 复制一份
    drive_db = '/content/drive/MyDrive/mcmod_data.db'
    if os.path.exists(drive_db):
        !cp "$drive_db" .
        print('已从 Drive 复制数据库。')
    else:
        print('未找到 mcmod_data.db，请手动上传到 Colab 文件系统！')
else:
    print('数据库已就绪。')

In [ ]:
# 加载数据
import sqlite3
import pandas as pd
from datasets import Dataset

conn = sqlite3.connect('mcmod_data.db')
df = pd.read_sql("SELECT prompt, code, mc_version, mod_loader FROM training_data", conn)
conn.close()

print(f"共加载 {len(df)} 条训练数据")

# 预览一条
df.head(1)

In [ ]:
# 格式化为模型可理解的文本
def format_example(row):
    system = f"[MC {row['mc_version']}] [{row['mod_loader']}]"
    return f"<|system|>\n{system}\n<|user|>\n{row['prompt']}\n<|assistant|>\n{row['code']}"

dataset = Dataset.from_pandas(df[['prompt','code','mc_version','mod_loader']])
dataset = dataset.map(lambda x: {"text": format_example(x)})

print("样本示例:")
print(dataset[0]["text"][:300])

In [ ]:
# 加载预训练代码模型
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Salesforce/codegen-350M-mono"  # 350M 参数，适合免费 T4 GPU
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)

print("模型加载完成")

In [ ]:
# 分词数据集
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
print(f"数据集大小: {len(tokenized_dataset)}")

In [ ]:
# 使用 LoRA 高效微调
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # CodeGen 的注意力模块
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 训练配置
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./mcmod-model",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,     # 有效 batch size = 8
    num_train_epochs=3,
    logging_steps=50,
    save_steps=200,
    learning_rate=2e-4,
    fp16=True,                        # 混合精度，加快训练并节省显存
    save_total_limit=2,
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer
)

print("开始训练...")
trainer.train()

In [ ]:
# 保存模型到 Google Drive
save_path = "/content/drive/MyDrive/mcmod-model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✅ 模型已保存至 {save_path}")

In [ ]:
# (可选) 直接下载模型到本地电脑
import shutil
shutil.make_archive('/content/mcmod-model', 'zip', '/content/drive/MyDrive/mcmod-model')
from google.colab import files
files.download('/content/mcmod-model.zip')

## 🧪 快速测试生成效果
训练完成后，运行下面单元格试试看

In [ ]:
def generate_mod(description, mc_version="1.20.1", mod_loader="Forge"):
    prompt = f"<|system|>\n[MC {mc_version}] [{mod_loader}]\n<|user|>\n创建一个{description}\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.8,
        do_sample=True,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result.split("<|assistant|>")[-1]

print(generate_mod("一把能召唤闪电的钻石剑"))